# Medical Chatbot - End-to-End RAG Pipeline
### Groq Llama | ChromaDB (free) | LangChain >= 0.2

**Architecture:**
```
data/Medical_book.pdf
    |
    v
src/helper.py --> load_pdf_file -> text_split -> HuggingFace embeddings
    |
    v
ChromaDB (local, persistent)  <- free vector store, no API key
    |
    v
src/prompt.py --> system_prompt
    |
    v
Groq API --> llama-3.1-8b-instant (fast, generous free tier)
    |
    v
RAG Chain (LangChain retrieval + stuff-documents)
```

**Phases:**
1. Install dependencies
2. Mount Google Drive and configure API key
3. Create project directory structure
4. Write src/helper.py and src/prompt.py
5. Ingest PDF -> Embed -> Store in ChromaDB
6. Build RAG chain with Groq/Llama
7. Interactive Q&A demo
8. Save all phase outputs and zip


## Phase 1 - Install Dependencies

In [24]:
# Phase 1: Install all required packages
# LangChain >= 0.2 ecosystem with Groq and ChromaDB (free local vector store).
# sentence-transformers supplies free HuggingFace embeddings (384-dim MiniLM).

!pip install -q --upgrade \
    langchain\
    langchain-community \
    langchain-core\
    langchain-groq\
    langchain-huggingface \
    langchain-chroma \
    chromadb \
    sentence-transformers \
    pypdf\
    groq

print("[OK] All packages installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.6/113.6 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 173.8/173.8 kB 9.8 MB/s eta 0:00:00
[OK] All packages installed.


## Phase 2 - Mount Google Drive and Configure API Key

In [3]:
# Phase 2a: Mount Google Drive so we can access the Medical PDF.
# Option 2: If Medical_book.pdf is already inside /content/
pdf_path = "/content/Medical_book.pdf"

import os
if os.path.exists(pdf_path):
    print(f"[OK] Found PDF at: {pdf_path}")
else:
    print("[ERROR] Medical_book.pdf not found in /content/")


[OK] Found PDF at: /content/Medical_book.pdf


In [4]:
# Phase 2b: Load GROQ_API_KEY from Colab Secrets.
# Steps: Runtime -> Manage secrets -> Add key named GROQ_API_KEY
# Free tier at https://console.groq.com (no credit card required)

import os
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
print("[OK] GROQ_API_KEY loaded from Colab secrets.")


[OK] GROQ_API_KEY loaded from Colab secrets.


## Phase 3 - Create Project Directory Structure

In [5]:
# Phase 3: Recreate the original project layout inside /content.
# Mirrors the on-disk structure from the original zip:
#   /content/medical_chatbot/
#       data/           <- PDF files live here
#       src/            <- helper.py, prompt.py, __init__.py
#       chroma_store/   <- ChromaDB persistence directory
#       outputs/        <- phase output artefacts saved here

import os, pathlib

BASE       = pathlib.Path("/content/medical_chatbot")
DATA_DIR   = BASE / 'data'
SRC_DIR    = BASE / 'src'
CHROMA_DIR = BASE / 'chroma_store'
OUTPUT_DIR = BASE / 'outputs'

for d in [DATA_DIR, SRC_DIR, CHROMA_DIR, OUTPUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("[OK] Directory structure created:")
for d in [BASE, DATA_DIR, SRC_DIR, CHROMA_DIR, OUTPUT_DIR]:
    print(f"   {d}")


[OK] Directory structure created:
   /content/medical_chatbot
   /content/medical_chatbot/data
   /content/medical_chatbot/src
   /content/medical_chatbot/chroma_store
   /content/medical_chatbot/outputs


In [6]:
# Phase 3b: Copy or upload Medical_book.pdf.
# Option A (preferred): PDF is in your Google Drive at the path below.
# Option B            : Uncomment the upload widget if not using Drive.

import shutil, pathlib

# -- Option A: copy from Drive (adjust path as needed) --
DRIVE_PDF = pathlib.Path("/content/drive/MyDrive/Medical_book.pdf")

if DRIVE_PDF.exists():
    shutil.copy(DRIVE_PDF, DATA_DIR / 'Medical_book.pdf')
    print(f"[OK] PDF copied from Drive -> {DATA_DIR / 'Medical_book.pdf'}")
else:
    print("[WARN] PDF not found on Drive. Launching upload widget...")
    from google.colab import files
    uploaded = files.upload()   # select your PDF
    for fname, data in uploaded.items():
        dest = DATA_DIR / fname
        dest.write_bytes(data)
        print(f"[OK] Uploaded {fname} -> {dest}")


[WARN] PDF not found on Drive. Launching upload widget...


Saving Medical_book.pdf to Medical_book (1).pdf
[OK] Uploaded Medical_book (1).pdf -> /content/medical_chatbot/data/Medical_book (1).pdf


## Phase 4 - Write src/helper.py and src/prompt.py

In [7]:
# Phase 4a: Create src/__init__.py to make src a Python package.

(SRC_DIR / "__init__.py").write_text("# Medical Chatbot src package\n")
print("[OK] src/__init__.py written.")


[OK] src/__init__.py written.


In [13]:
helper_src = '''
# src/helper.py
# Compatible with LangChain >= 0.2

from typing import List

# LangChain >= 0.2: document loaders in langchain-community
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader

# Core text splitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

# LangChain >= 0.2: HuggingFace embeddings in dedicated sub-package
from langchain_huggingface import HuggingFaceEmbeddings

# Schema
from langchain_core.documents import Document


def load_pdf_file(data: str) -> List[Document]:
    """Load all PDF files from the given directory path."""
    loader = DirectoryLoader(
        data,
        glob="*.pdf",
        loader_cls=PyPDFLoader,
    )
    return loader.load()


def filter_to_minimal_docs(docs: List[Document]) -> List[Document]:
    """Strip all metadata except source to avoid ChromaDB serialisation issues."""
    minimal_docs: List[Document] = []
    for doc in docs:
        src = doc.metadata.get("source", "unknown")
        minimal_docs.append(
            Document(
                page_content=doc.page_content,
                metadata={"source": src},
            )
        )
    return minimal_docs


def text_split(extracted_data: List[Document]) -> List[Document]:
    """Split documents into 500-char chunks with 20-char overlap."""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=20,
    )
    return text_splitter.split_documents(extracted_data)


def download_hugging_face_embeddings() -> HuggingFaceEmbeddings:
    """Load sentence-transformers/all-MiniLM-L6-v2 (384 dims, free, no API key)."""
    return HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )
'''

(SRC_DIR / 'helper.py').write_text(helper_src.strip())
print("[OK] src/helper.py written.")

[OK] src/helper.py written.


In [9]:
# Phase 4c: Write src/prompt.py
# Maintains the exact variable name `system_prompt` used in the original app.py.
# {context} is filled by LangChain's create_stuff_documents_chain at runtime.

prompt_src = '''
# src/prompt.py
# System prompt for the Medical Chatbot RAG chain.
# {context} is replaced with retrieved document chunks at runtime.

system_prompt = (
    "You are a Medical assistant for question-answering tasks. "
    "Use ONLY the following pieces of retrieved context to answer the question. "
    "If you do not know the answer, say that you do not know. "
    "Use three sentences maximum and keep the answer concise.\\n\\n"
    "{context}"
)
'''

(SRC_DIR / 'prompt.py').write_text(prompt_src.strip())
print("[OK] src/prompt.py written.")


[OK] src/prompt.py written.


In [10]:
# Phase 4d: Verify src package structure

import os
for root, dirs, files in os.walk(SRC_DIR):
    for f in files:
        print(os.path.join(root, f))


/content/medical_chatbot/src/__init__.py
/content/medical_chatbot/src/prompt.py
/content/medical_chatbot/src/helper.py


## Phase 5 - Ingest PDF, Embed, Store in ChromaDB

In [11]:
# Phase 5a: Add project root to sys.path so `src` is importable.

import sys
sys.path.insert(0, str(BASE))
print(f"[OK] sys.path updated with {BASE}")


[OK] sys.path updated with /content/medical_chatbot


In [14]:
# Phase 5b: Load PDF documents using src/helper.py

from src.helper import (
    load_pdf_file,
    filter_to_minimal_docs,
    text_split,
    download_hugging_face_embeddings,
)

extracted_data = load_pdf_file(data=str(DATA_DIR))
print(f"[OK] Loaded {len(extracted_data)} pages from PDF(s).")

# Save phase output
with open(OUTPUT_DIR / 'phase5_load_summary.txt', 'w') as f:
    f.write(f'Total pages loaded: {len(extracted_data)}\n')
    for i, doc in enumerate(extracted_data[:5]):
        f.write(f'\n--- Page {i+1} preview ---\n')
        f.write(doc.page_content[:300] + '\n')
print("[OK] Load summary saved to outputs/phase5_load_summary.txt")

[OK] Loaded 637 pages from PDF(s).
[OK] Load summary saved to outputs/phase5_load_summary.txt


In [15]:
# Phase 5c: Filter metadata to only keep 'source'.
# Prevents ChromaDB serialisation errors from complex metadata
# that PyPDFLoader sometimes produces (dates, producer strings, etc.).

filter_data = filter_to_minimal_docs(extracted_data)
print(f"[OK] Metadata filtered. Sample: {filter_data[0].metadata}")


[OK] Metadata filtered. Sample: {'source': '/content/medical_chatbot/data/Medical_book (1).pdf'}


In [16]:
# Phase 5d: Split documents into 500-char overlapping chunks.
# Keeps each chunk well within any LLM context window
# and gives the retriever fine-grained control over returned context.

text_chunks = text_split(filter_data)
print(f"[OK] Split into {len(text_chunks)} text chunks.")
print(f"   Sample chunk: {text_chunks[0].page_content[:200]}")

with open(OUTPUT_DIR / 'phase5_chunks_summary.txt', 'w') as f:
    f.write(f'Total chunks: {len(text_chunks)}\n\n')
    for i, chunk in enumerate(text_chunks[:10]):
        f.write(f'--- Chunk {i+1} ---\n{chunk.page_content[:300]}\n\n')
print("[OK] Chunks summary saved to outputs/phase5_chunks_summary.txt")


[OK] Split into 5860 text chunks.
   Sample chunk: The GALE
ENCYCLOPEDIA
of MEDICINE
SECOND EDITION
[OK] Chunks summary saved to outputs/phase5_chunks_summary.txt


In [18]:
# Phase 5e: Load HuggingFace MiniLM embeddings (384-dim, free, no API key).

embeddings = download_hugging_face_embeddings()
print("[OK] HuggingFace embeddings loaded (all-MiniLM-L6-v2, 384 dims).")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[OK] HuggingFace embeddings loaded (all-MiniLM-L6-v2, 384 dims).


In [19]:
# Phase 5f: Build and persist ChromaDB vector store.
# ChromaDB is fully local, free, and open-source.
# persist_directory keeps the index across Colab sessions (on Drive if needed).
#
# Token-limit note: MiniLM embeds locally so there are no remote API
# token limits to worry about during the embedding phase.

from langchain_chroma import Chroma

CHROMA_COLLECTION = 'medical_chatbot'

vectorstore = Chroma.from_documents(
    documents=text_chunks,
    embedding=embeddings,
    collection_name=CHROMA_COLLECTION,
    persist_directory=str(CHROMA_DIR),
)

print(f"[OK] ChromaDB vector store created with {len(text_chunks)} vectors.")
print(f"   Persist directory: {CHROMA_DIR}")

with open(OUTPUT_DIR / 'phase5_vectorstore_summary.txt', 'w') as f:
    f.write(f'Collection : {CHROMA_COLLECTION}\n')
    f.write(f'Persist dir: {CHROMA_DIR}\n')
    f.write(f'Vectors    : {len(text_chunks)}\n')
    f.write('Embedding  : sentence-transformers/all-MiniLM-L6-v2 (384 dims)\n')
print("[OK] VectorStore summary saved to outputs/phase5_vectorstore_summary.txt")


[OK] ChromaDB vector store created with 5860 vectors.
   Persist directory: /content/medical_chatbot/chroma_store
[OK] VectorStore summary saved to outputs/phase5_vectorstore_summary.txt


## Phase 6 - Build RAG Chain with Groq / Llama

In [20]:
# Phase 6a: Initialise the Groq LLM.
# Model  : llama-3.1-8b-instant
#   Free tier : 6,000 tokens/min, 500,000 tokens/day
#   Context   : 128,000 tokens
#   Speed     : ~250 tokens/sec
#
# Token-limit compliance strategy:
#   - Retriever returns k=3 chunks of 500 chars each (~375 tokens context)
#   - System prompt is concise (~50 tokens)
#   - max_tokens capped at 512 to stay safely within the 6k TPM free limit

from langchain_groq import ChatGroq

llm = ChatGroq(
    model='llama-3.1-8b-instant',
    temperature=0.2,           # low temp -> factual, grounded answers
    max_tokens=512,            # capped to respect 6,000 TPM free limit
    groq_api_key=os.environ["GROQ_API_KEY"],
)
print("[OK] Groq LLM initialised: llama-3.1-8b-instant (max_tokens=512).")


[OK] Groq LLM initialised: llama-3.1-8b-instant (max_tokens=512).


In [21]:
# Phase 6b: Reload ChromaDB from disk and create the similarity retriever.

from langchain_chroma import Chroma

vectorstore = Chroma(
    collection_name=CHROMA_COLLECTION,
    embedding_function=embeddings,
    persist_directory=str(CHROMA_DIR),
)

# k=3: top-3 most similar 500-char chunks (~375 tokens of context)
retriever = vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 3},
)
print("[OK] ChromaDB retriever ready (top-k=3).")


[OK] ChromaDB retriever ready (top-k=3).


In [27]:
# LangChain 1.2+ modular imports (new package structure)

from langchain_classic.chains.retrieval import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate
from src.prompt import system_prompt   # original prompt file

# Build prompt template
prompt_template = ChatPromptTemplate.from_messages([
    ("system", system_prompt),   # {context} auto-filled from retrieved docs
    ("human", "{input}"),
])

# QA chain
question_answer_chain = create_stuff_documents_chain(
    llm,
    prompt_template
)

# Full retrieval chain
rag_chain = create_retrieval_chain(
    retriever,
    question_answer_chain
)

print("[OK] RAG chain built successfully.")
print("   Retriever : ChromaDB similarity (k=3)")
print("   LLM       : Groq llama-3.1-8b-instant")
print("   Prompt    : src/prompt.py -> system_prompt")

[OK] RAG chain built successfully.
   Retriever : ChromaDB similarity (k=3)
   LLM       : Groq llama-3.1-8b-instant
   Prompt    : src/prompt.py -> system_prompt


In [28]:
# Phase 6d: Save RAG chain config to outputs.

with open(OUTPUT_DIR / 'phase6_rag_chain_config.txt', 'w') as f:
    f.write('RAG Chain Configuration\n')
    f.write('=' * 40 + '\n')
    f.write('LLM          : Groq / llama-3.1-8b-instant\n')
    f.write('Temperature  : 0.2\n')
    f.write('Max tokens   : 512\n')
    f.write('Retriever    : ChromaDB similarity, k=3\n')
    f.write('Chunk size   : 500 chars / 20 overlap\n')
    f.write('Embedding    : all-MiniLM-L6-v2 (384 dims)\n')
    f.write('System prompt: src/prompt.py\n')
print("[OK] RAG chain config saved to outputs/phase6_rag_chain_config.txt")


[OK] RAG chain config saved to outputs/phase6_rag_chain_config.txt


## Phase 7 - Interactive Q&A Demo

In [29]:
# Phase 7a: Helper function to ask a question and display the answer + sources.

def ask(question: str) -> str:
    """
    Run a question through the RAG chain and return the answer.
    Prints the answer and the retrieved source chunks for transparency.
    """
    response = rag_chain.invoke({'input': question})
    answer   = response['answer']
    context  = response.get('context', [])

    print(f'\nQuestion : {question}')
    print(f'Answer   : {answer}')
    print(f'\nSources ({len(context)} chunks retrieved):')
    for i, doc in enumerate(context, 1):
        src     = doc.metadata.get('source', 'unknown')
        preview = doc.page_content[:120].replace('\n', ' ')
        print(f'   [{i}] {src} - "{preview}..."')
    return answer

print("[OK] ask() helper defined.")


[OK] ask() helper defined.


In [30]:
# Phase 7b: Run sample questions and save results.

sample_questions = [
    'What is diabetes and what are its main types?',
    'What are common symptoms of hypertension?',
    'How is pneumonia diagnosed and treated?',
]

qa_log = []
for q in sample_questions:
    ans = ask(q)
    qa_log.append({'question': q, 'answer': ans})

with open(OUTPUT_DIR / 'phase7_qa_demo_output.txt', 'w') as f:
    f.write('Medical Chatbot - Sample Q&A Demo Output\n')
    f.write('=' * 50 + '\n\n')
    for item in qa_log:
        f.write(f"Q: {item['question']}\n")
        f.write(f"A: {item['answer']}\n\n")
print("\n[OK] Q&A demo output saved to outputs/phase7_qa_demo_output.txt")



Question : What is diabetes and what are its main types?
Answer   : Diabetes mellitus is a disorder of carbohydrate metabolism brought on by a combination of hereditary and environmental factors. It can be divided into two main types: type I, formerly termed juvenile onset or insulin-dependent, and type II.

Sources (3 chunks retrieved):
   [1] /content/medical_chatbot/data/Medical_book (1).pdf - "Resources BOOKS Berkow, Robert, ed. The Merck Manual of Medical Informa- tion: Home Edition. Whitehouse Station, NJ: Mer..."
   [2] /content/medical_chatbot/data/Medical_book (1).pdf - "Resources BOOKS A Manual of Laboratory and Diagnostic Tests.5th ed. Ed. Francis Fishback. Philadelphia: Lippincott, 1996..."
   [3] /content/medical_chatbot/data/Medical_book (1).pdf - "with a physician or pharmacist before combining tri- cyclic antidepressants with any other prescription or non- prescrip..."

Question : What are common symptoms of hypertension?
Answer   : High blood pressure can cause variou

In [31]:
# Phase 7c: Single interactive question cell.
# Change YOUR_QUESTION and re-run this cell to query the chatbot.

YOUR_QUESTION = 'What are the causes and risk factors of heart disease?'
_ = ask(YOUR_QUESTION)



Question : What are the causes and risk factors of heart disease?
Answer   : The causes and risk factors of heart disease include atherosclerosis, which is often caused by heart attacks, and is increased by diabetes, obesity, and heredity. Other risk factors include high blood pressure and certain types of arrhythmias. Additionally, slow or fast heart rates can also be a risk factor for heart disease.

Sources (3 chunks retrieved):
   [1] /content/medical_chatbot/data/Medical_book (1).pdf - "lowered by keeping diabetes under control. Most dia- betics die from heart attacks caused by atherosclerosis. • Obesity—..."
   [2] /content/medical_chatbot/data/Medical_book (1).pdf - "people with heart disease, it is usually the heart disease which is dangerous, not the arrhythmia. Arrhythmias often occ..."
   [3] /content/medical_chatbot/data/Medical_book (1).pdf - "Heart Attacks” and “Diseases of the Peripheral Arteries and Veins.” In Texas Heart Institute Heart Owner’s Hand- book. N..."


## Phase 8 - Save All Phase Outputs and Zip

In [32]:
# Phase 8a: List all output artefacts.

import os
print("[DIR] Contents of outputs/ directory:")
for f in sorted(os.listdir(OUTPUT_DIR)):
    size = os.path.getsize(OUTPUT_DIR / f)
    print(f"   {f:45s} {size:>8,} bytes")


[DIR] Contents of outputs/ directory:
   phase5_chunks_summary.txt                        2,799 bytes
   phase5_load_summary.txt                            927 bytes
   phase5_vectorstore_summary.txt                     161 bytes
   phase6_rag_chain_config.txt                        296 bytes
   phase7_qa_demo_output.txt                        1,049 bytes


In [33]:
# Phase 8b: Copy src files into outputs for completeness.

import shutil
shutil.copy(SRC_DIR / 'helper.py', OUTPUT_DIR / 'src_helper.py')
shutil.copy(SRC_DIR / 'prompt.py', OUTPUT_DIR / 'src_prompt.py')
print("[OK] src/helper.py and src/prompt.py copied to outputs/")


[OK] src/helper.py and src/prompt.py copied to outputs/


In [34]:
# Phase 8c: Write project README to outputs.

readme = """# Medical Chatbot - Project Summary

## Stack (all free / open-source)
- LLM       : Groq llama-3.1-8b-instant (free tier)
- Vector DB : ChromaDB (local, free, no API key)
- Embeddings: HuggingFace all-MiniLM-L6-v2 (free, 384 dims)
- Framework : LangChain >= 0.2
- PDF Loader: PyPDFLoader (langchain-community)

## Token Limit Compliance (Groq free tier)
- Free limit  : 6,000 tokens/min, 500,000 tokens/day
- Retriever k : 3 chunks x 500 chars = ~375 context tokens
- System prompt: ~50 tokens
- max_tokens  : 512 (capped in ChatGroq init)
- Estimated per-call: ~950 tokens (well within limits)

## Output Files
- phase5_load_summary.txt     : PDF loading summary
- phase5_chunks_summary.txt   : Text chunk details
- phase5_vectorstore_summary.txt : ChromaDB config
- phase6_rag_chain_config.txt : RAG chain parameters
- phase7_qa_demo_output.txt   : Sample Q&A responses
- src_helper.py               : src/helper.py (LangChain >= 0.2)
- src_prompt.py               : src/prompt.py (system prompt)
"""

(OUTPUT_DIR / 'README.md').write_text(readme)
print("[OK] README.md written to outputs/")


[OK] README.md written to outputs/


In [35]:
# Phase 8d: Create the final zip archive of all outputs.

import zipfile

ZIP_PATH = BASE / 'medical_chatbot_outputs.zip'

with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in sorted(OUTPUT_DIR.iterdir()):
        zf.write(f, arcname=f'outputs/{f.name}')
    zf.writestr(
        'chroma_store_note.txt',
        f'ChromaDB persisted at: {CHROMA_DIR}\n'
        'Copy this folder to restore the vector index without re-embedding.\n'
    )

print(f"[OK] Zip created: {ZIP_PATH}")
print(f"   Size: {ZIP_PATH.stat().st_size:,} bytes")


[OK] Zip created: /content/medical_chatbot/medical_chatbot_outputs.zip
   Size: 5,538 bytes


In [36]:
# Phase 8e: Download the zip to your local machine.

from google.colab import files
files.download(str(ZIP_PATH))
print("[OK] Download initiated for medical_chatbot_outputs.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

[OK] Download initiated for medical_chatbot_outputs.zip


---
## Notebook Complete

| Phase | Description | Key Output |
|-------|-------------|------------|
| 1 | Install dependencies | - |
| 2 | Drive mount + GROQ API key | - |
| 3 | Project directory structure | /content/medical_chatbot/ |
| 4 | Write src/helper.py and src/prompt.py | src/ |
| 5 | PDF ingest -> embed -> ChromaDB | chroma_store/ + 3 txt files |
| 6 | Build RAG chain (Groq Llama) | config txt |
| 7 | Interactive Q&A demo | phase7_qa_demo_output.txt |
| 8 | Save + zip all outputs | medical_chatbot_outputs.zip |

**To ask your own question:** change `YOUR_QUESTION` in Phase 7c and re-run that cell.
